In [69]:
import pandas as pd
import numpy as np

# Đọc dữ liệu

In [70]:
raw_data = pd.read_csv('C:\\Users\\User\\Desktop\\Project\\uk_train_transaction.csv')

In [71]:
raw_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 31653 entries, 0 to 31652
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Transaction_ID        31653 non-null  str  
 1   Date_of_Purchase      31653 non-null  str  
 2   Time_of_Purchase      31653 non-null  str  
 3   Purchase_Type         31653 non-null  str  
 4   Payment_Method        31653 non-null  str  
 5   Price                 31653 non-null  int64
 6   Departure_Station     31653 non-null  str  
 7   Arrival_Destination   31653 non-null  str  
 8   Date_of_Journey       31653 non-null  str  
 9   Departure_Time        31653 non-null  str  
 10  Arrival_Time          31653 non-null  str  
 11  Actual_Arrival_Time   29773 non-null  str  
 12  Journey_Status        31653 non-null  str  
 13  Reason_for_Delay      4172 non-null   str  
 14  Refund_Request        31653 non-null  str  
 15  combined_ticket_info  31653 non-null  str  
dtypes: int64(1), st

In [72]:
raw_data.head()

,Transaction_ID,Date_of_Purchase,Time_of_Purchase,Purchase_Type,Payment_Method,Price,Departure_Station,Arrival_Destination,Date_of_Journey,Departure_Time,Arrival_Time,Actual_Arrival_Time,Journey_Status,Reason_for_Delay,Refund_Request,combined_ticket_info
0,da8a6ba8-b3dc-4677-b176,12/8/2023,12:41:11,Online,Contactless,43,London Paddington,Liverpool Lime Street,1/1/2024,11:00:00,13:30:00,13:30:00,On Time,NaN,No,"{""ticket_class"": ""Standard"", ""ticket_type"": ""A..."
1,b0cdd1b0-f214-4197-be53,12/16/2023,11:23:01,Station,Credit Card,23,London Kings Cross,York,1/1/2024,9:45:00,11:35:00,11:40:00,Delayed,Signal Failure,No,"{""ticket_class"": ""Standard"", ""ticket_type"": ""A..."
2,f3ba7a96-f713-40d9-9629,12/19/2023,19:51:27,Online,Credit Card,3,Liverpool Lime Street,Manchester Piccadilly,1/2/2024,18:15:00,18:45:00,18:45:00,On Time,NaN,No,"{""ticket_class"": ""Standard"", ""ticket_type"": ""A..."
3,b2471f11-4fe7-4c87-8ab4,12/20/2023,23:00:36,Station,Credit Card,13,London Paddington,Reading,1/1/2024,21:30:00,22:30:00,22:30:00,On Time,NaN,No,"{""ticket_class"": ""Standard"", ""ticket_type"": ""A..."
4,2be00b45-0762-485e-a7a3,12/27/2023,18:22:56,Online,Contactless,76,Liverpool Lime Street,London Euston,1/1/2024,16:45:00,19:00:00,19:00:00,On Time,NaN,No,"{""ticket_class"": ""Standard"", ""ticket_type"": ""A..."


**Vấn đề của bộ dữ liệu**
* Sai kiểu dữ liệu:
    * Thời gian: Date_of_Purchase, Time_of_Purchase, Date_of_Journey, Departure_Time, Arrival_Time, Actual_Arrival_Time
* Giá trị NULL do biểu diễn dữ liệu (Actual_Arrival_Time: chuyến xe bị hủy, Reason_for_Delay: chuyến xe không bị Delay)
* Dữ liệu đang biểu diễn combined
    

# Tiền xử lý dữ liệu

In [73]:
cols_date = ['Date_of_Purchase', 'Date_of_Journey']
raw_data[cols_date] = raw_data[cols_date].apply(pd.to_datetime, errors='coerce')


In [74]:
time_cols = ['Time_of_Purchase', 'Departure_Time', 'Arrival_Time', 'Actual_Arrival_Time']
raw_data[time_cols] = raw_data[time_cols].apply(lambda col: pd.to_datetime(col, format='%H:%M:%S', errors='coerce').dt.time)

In [75]:
import ast
raw_data['combined_ticket_info'] = raw_data['combined_ticket_info'].apply(ast.literal_eval)
tempt_data = pd.json_normalize(raw_data['combined_ticket_info'])
raw_data = pd.concat([raw_data.drop(columns=['combined_ticket_info']), tempt_data], axis=1)

In [76]:
raw_data.head()

,Transaction_ID,Date_of_Purchase,Time_of_Purchase,Purchase_Type,Payment_Method,Price,Departure_Station,Arrival_Destination,Date_of_Journey,Departure_Time,Arrival_Time,Actual_Arrival_Time,Journey_Status,Reason_for_Delay,Refund_Request,ticket_class,ticket_type,railcard
0,da8a6ba8-b3dc-4677-b176,2023-12-08,12:41:11,Online,Contactless,43,London Paddington,Liverpool Lime Street,2024-01-01,11:00:00,13:30:00,13:30:00,On Time,NaN,No,Standard,Advance,Adult
1,b0cdd1b0-f214-4197-be53,2023-12-16,11:23:01,Station,Credit Card,23,London Kings Cross,York,2024-01-01,09:45:00,11:35:00,11:40:00,Delayed,Signal Failure,No,Standard,Advance,Adult
2,f3ba7a96-f713-40d9-9629,2023-12-19,19:51:27,Online,Credit Card,3,Liverpool Lime Street,Manchester Piccadilly,2024-01-02,18:15:00,18:45:00,18:45:00,On Time,NaN,No,Standard,Advance,No info
3,b2471f11-4fe7-4c87-8ab4,2023-12-20,23:00:36,Station,Credit Card,13,London Paddington,Reading,2024-01-01,21:30:00,22:30:00,22:30:00,On Time,NaN,No,Standard,Advance,No info
4,2be00b45-0762-485e-a7a3,2023-12-27,18:22:56,Online,Contactless,76,Liverpool Lime Street,London Euston,2024-01-01,16:45:00,19:00:00,19:00:00,On Time,NaN,No,Standard,Advance,No info


In [77]:
raw_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 31653 entries, 0 to 31652
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Transaction_ID       31653 non-null  str           
 1   Date_of_Purchase     31653 non-null  datetime64[us]
 2   Time_of_Purchase     31653 non-null  object        
 3   Purchase_Type        31653 non-null  str           
 4   Payment_Method       31653 non-null  str           
 5   Price                31653 non-null  int64         
 6   Departure_Station    31653 non-null  str           
 7   Arrival_Destination  31653 non-null  str           
 8   Date_of_Journey      31653 non-null  datetime64[us]
 9   Departure_Time       31653 non-null  object        
 10  Arrival_Time         31653 non-null  object        
 11  Actual_Arrival_Time  29773 non-null  object        
 12  Journey_Status       31653 non-null  str           
 13  Reason_for_Delay     4172 non-null   str  

In [78]:
raw_data.fillna({'Reason_for_Delay': 'No Delay'}, inplace=True)

,Transaction_ID,Date_of_Purchase,Time_of_Purchase,Purchase_Type,Payment_Method,Price,Departure_Station,Arrival_Destination,Date_of_Journey,Departure_Time,Arrival_Time,Actual_Arrival_Time,Journey_Status,Reason_for_Delay,Refund_Request,ticket_class,ticket_type,railcard
0,da8a6ba8-b3dc-4677-b176,2023-12-08,12:41:11,Online,Contactless,43,London Paddington,Liverpool Lime Street,2024-01-01,11:00:00,13:30:00,13:30:00,On Time,No Delay,No,Standard,Advance,Adult
1,b0cdd1b0-f214-4197-be53,2023-12-16,11:23:01,Station,Credit Card,23,London Kings Cross,York,2024-01-01,09:45:00,11:35:00,11:40:00,Delayed,Signal Failure,No,Standard,Advance,Adult
2,f3ba7a96-f713-40d9-9629,2023-12-19,19:51:27,Online,Credit Card,3,Liverpool Lime Street,Manchester Piccadilly,2024-01-02,18:15:00,18:45:00,18:45:00,On Time,No Delay,No,Standard,Advance,No info
3,b2471f11-4fe7-4c87-8ab4,2023-12-20,23:00:36,Station,Credit Card,13,London Paddington,Reading,2024-01-01,21:30:00,22:30:00,22:30:00,On Time,No Delay,No,Standard,Advance,No info
4,2be00b45-0762-485e-a7a3,2023-12-27,18:22:56,Online,Contactless,76,Liverpool Lime Street,London Euston,2024-01-01,16:45:00,19:00:00,19:00:00,On Time,No Delay,No,Standard,Advance,No info
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31648,1304623d-b8b7-4999-8e9c,2024-04-30,18:42:58,Online,Credit Card,4,Manchester Piccadilly,Liverpool Lime Street,2024-04-30,20:00:00,20:30:00,20:30:00,On Time,No Delay,No,Standard,Off-Peak,No info
31649,7da22246-f480-417c-bc2f,2024-04-30,18:46:10,Online,Contactless,10,London Euston,Birmingham New Street,2024-04-30,20:15:00,21:35:00,21:35:00,On Time,No Delay,No,Standard,Off-Peak,No info
31650,add9debf-46c1-4c75-b52d,2024-04-30,18:56:41,Station,Credit Card,4,Manchester Piccadilly,Liverpool Lime Street,2024-04-30,20:15:00,20:45:00,20:45:00,On Time,No Delay,No,Standard,Off-Peak,No info
31651,b92b047c-21fd-4859-966a,2024-04-30,19:51:47,Station,Credit Card,10,London Euston,Birmingham New Street,2024-04-30,21:15:00,22:35:00,22:35:00,On Time,No Delay,No,Standard,Off-Peak,No info


In [79]:
mapping_dict = {
    'Signal failure': 'Signal Failure',
    'Weather': 'Weather Conditions'
}
raw_data['Reason_for_Delay'] = raw_data['Reason_for_Delay'].replace(mapping_dict)

In [80]:
raw_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 31653 entries, 0 to 31652
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Transaction_ID       31653 non-null  str           
 1   Date_of_Purchase     31653 non-null  datetime64[us]
 2   Time_of_Purchase     31653 non-null  object        
 3   Purchase_Type        31653 non-null  str           
 4   Payment_Method       31653 non-null  str           
 5   Price                31653 non-null  int64         
 6   Departure_Station    31653 non-null  str           
 7   Arrival_Destination  31653 non-null  str           
 8   Date_of_Journey      31653 non-null  datetime64[us]
 9   Departure_Time       31653 non-null  object        
 10  Arrival_Time         31653 non-null  object        
 11  Actual_Arrival_Time  29773 non-null  object        
 12  Journey_Status       31653 non-null  str           
 13  Reason_for_Delay     31653 non-null  str  

In [81]:
date_cols = ['Date_of_Purchase', 'Date_of_Journey']
time_cols = ['Arrival_Time', 'Actual_Arrival_Time', 'Time_of_Purchase', 'Departure_Time']

for cols in raw_data.columns:
    if cols in date_cols:
        raw_data[cols] = raw_data[cols].apply(pd.to_datetime, errors='coerce')
    elif cols in time_cols:
        raw_data[cols] = raw_data[cols].apply(lambda col: pd.to_datetime(col, format='%H:%M:%S', errors='coerce').time() if pd.notna(col) else None)

In [82]:
from pandas.api.types import CategoricalDtype

part_of_day = {
    'Morning' : (pd.to_timedelta('05:00:00'), pd.to_timedelta('11:59:59')),
    'Afternoon' : (pd.to_timedelta('12:00:00'), pd.to_timedelta('16:59:59')),
    'Evening' : (pd.to_timedelta('17:00:00'), pd.to_timedelta('21:59:59')),
    'Night' : (pd.to_timedelta('22:00:00'), pd.to_timedelta('04:59:59'))
}

part_of_day_dtype = CategoricalDtype(categories=[*part_of_day.keys(), 'Unknown'], ordered=True)

def assign_part_of_day(time):
    if pd.isna(time):
        return 'Unknown'
    time_td = pd.to_timedelta(time.strftime('%H:%M:%S'))
    for part, (start, end) in part_of_day.items():
        if start <= end:
            if start <= time_td <= end:
                return part
        else:
            if time_td >= start or time_td <= end:
                return part
    return 'Unknown'

raw_data['Part_of_Day'] = raw_data['Departure_Time'].apply(assign_part_of_day).astype(part_of_day_dtype)


In [83]:
raw_data['month_year'] = pd.to_datetime(raw_data['Date_of_Purchase']).dt.to_period('M')

In [84]:
bins = [0, 15, 40, 90, np.inf]
labels = ['Low', 'Medium', 'Medium High', 'High']

raw_data['Price_category'] = pd.cut(raw_data['Price'], bins=bins, labels=labels, include_lowest=True)
raw_data['ticket_price_class'] = raw_data['ticket_class'].astype(str)+'-'+raw_data['Price_category'].astype(str)

In [85]:
raw_data['lead_time'] = (raw_data['Date_of_Journey'] - raw_data['Date_of_Purchase']).dt.days

In [86]:
raw_data['time_journey'] = pd.to_timedelta(raw_data['Arrival_Time'].astype(str)) - pd.to_timedelta(raw_data['Departure_Time'].astype(str))
raw_data['time_journey'] = raw_data['time_journey'].dt.total_seconds() / 60
raw_data['time_journey'] = raw_data['time_journey'].apply(lambda x: x + 24*60 if x < -12*60 else x)

In [87]:
bins = [0, 60, 120, 180, np.inf]
labels = ['Quick Trip', 'Short Trip', 'Medium Trip', 'Long Trip']

raw_data['Journey_Type'] = pd.cut(raw_data['time_journey'], bins=bins, labels=labels, include_lowest=True)

In [88]:
# delay_time = raw_data.dropna(subset=['Actual_Arrival_Time', 'Arrival_Time']).copy()
# delay_time = delay_time.loc[delay_time['Reason_for_Delay'] != 'No Delay', :]

# actual_td = pd.to_timedelta(delay_time['Actual_Arrival_Time'].astype(str))
# arrival_td = pd.to_timedelta(delay_time['Arrival_Time'].astype(str))

# delay_time['Delay_Time'] = (actual_td - arrival_td).dt.total_seconds() / 60
# delay_time.loc[delay_time['Delay_Time'] < -720, 'Delay_Time'] += 1440

In [89]:
raw_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 31653 entries, 0 to 31652
Data columns (total 25 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Transaction_ID       31653 non-null  str           
 1   Date_of_Purchase     31653 non-null  datetime64[us]
 2   Time_of_Purchase     31653 non-null  object        
 3   Purchase_Type        31653 non-null  str           
 4   Payment_Method       31653 non-null  str           
 5   Price                31653 non-null  int64         
 6   Departure_Station    31653 non-null  str           
 7   Arrival_Destination  31653 non-null  str           
 8   Date_of_Journey      31653 non-null  datetime64[us]
 9   Departure_Time       31653 non-null  object        
 10  Arrival_Time         31653 non-null  object        
 11  Actual_Arrival_Time  29773 non-null  object        
 12  Journey_Status       31653 non-null  str           
 13  Reason_for_Delay     31653 non-null  str  

In [92]:
actual_td = pd.to_timedelta(raw_data['Actual_Arrival_Time'].astype(str), errors='coerce')
arrival_td = pd.to_timedelta(raw_data['Arrival_Time'].astype(str), errors='coerce')

raw_data['Delay_Time'] = (actual_td - arrival_td).dt.total_seconds() / 60

# 3. Xử lý logic qua đêm (Chuyến đi dự kiến 23:50, đến lúc 00:10 -> bị tính âm 1420 phút)
raw_data.loc[raw_data['Delay_Time'] < -720, 'Delay_Time'] += 1440

# Những chuyến tàu chạy đúng giờ (No Delay) nên  Delay_Time là 0 phút thay vì để Null
raw_data.loc[raw_data['Reason_for_Delay'] == 'No Delay', 'Delay_Time'] = 0

In [93]:
raw_data.head()

,Transaction_ID,Date_of_Purchase,Time_of_Purchase,Purchase_Type,Payment_Method,Price,Departure_Station,Arrival_Destination,Date_of_Journey,Departure_Time,...,ticket_type,railcard,Part_of_Day,month_year,Price_category,ticket_price_class,lead_time,time_journey,Journey_Type,Delay_Time
0,da8a6ba8-b3dc-4677-b176,2023-12-08,12:41:11,Online,Contactless,43,London Paddington,Liverpool Lime Street,2024-01-01,11:00:00,...,Advance,Adult,Morning,2023-12,Medium High,Standard-Medium High,24,150.0,Medium Trip,0.0
1,b0cdd1b0-f214-4197-be53,2023-12-16,11:23:01,Station,Credit Card,23,London Kings Cross,York,2024-01-01,09:45:00,...,Advance,Adult,Morning,2023-12,Medium,Standard-Medium,16,110.0,Short Trip,5.0
2,f3ba7a96-f713-40d9-9629,2023-12-19,19:51:27,Online,Credit Card,3,Liverpool Lime Street,Manchester Piccadilly,2024-01-02,18:15:00,...,Advance,No info,Evening,2023-12,Low,Standard-Low,14,30.0,Quick Trip,0.0
3,b2471f11-4fe7-4c87-8ab4,2023-12-20,23:00:36,Station,Credit Card,13,London Paddington,Reading,2024-01-01,21:30:00,...,Advance,No info,Evening,2023-12,Low,Standard-Low,12,60.0,Quick Trip,0.0
4,2be00b45-0762-485e-a7a3,2023-12-27,18:22:56,Online,Contactless,76,Liverpool Lime Street,London Euston,2024-01-01,16:45:00,...,Advance,No info,Afternoon,2023-12,Medium High,Standard-Medium High,5,135.0,Medium Trip,0.0


In [95]:
raw_data.to_csv('C:\\Users\\User\\Desktop\\Project\\uk_train_powerBI_cleaned.csv', index=False)